In [1]:
from os import listdir
from experiment_main import *
        
# ---------------------------------------------------------------------------------------
# Significance-test side channel (C7).
#
# deflated_sharpe needs two things that do not survive anywhere else: the SELECTED config's
# per-trade return series, and the sharpe of EVERY config that was searched (their spread
# sets the multiple-testing hurdle). naked_crossover computes the per-trade bets for all
# 475 configs on the validation window and get_crossover_results then discards them,
# keeping only the aggregates in dff_valid. This collects them instead.
#
# One record is appended per (crossover, system) run. Cleared by the caller before a run.
# ---------------------------------------------------------------------------------------
validation_bets = []


def get_crossover_results(
    df_test,
    df_valid,
    name_mode,
    target_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
):
    dff_test, current_bet_list_optim = naked_crossover(
        df_test,
        target_list,
        signal_line_list,
        trade_cost,
        balance,
        risk_per_trade=risk_per_trade,
        signal_line_if=signal_line_if,
        signal_line_if_target=signal_line_if_target,
        last_close=last_close,
    )

    check_index_first = find_best_index(
        dff_test, 1, "net", name_mode, alpha_range, gamma_range
    )
    check_index_second = find_best_index(
        dff_test, 2, "net", name_mode, alpha_range, gamma_range
    )
    check_index_third, best_list = find_best_index(
        dff_test, 3, "net", name_mode, alpha_range, gamma_range
    )
    check_index_fourth = find_best_index(
        dff_test, 4, "net", name_mode, alpha_range, gamma_range
    )  # Genetic algorithm
    # The long/short rule above picks a config by "net". A long-only trader would not pick
    # that config -- they would pick whichever config is best on LONG TRADES ALONE. So
    # select again on "long_only_net". This is what the main experiment already does
    # (experiment_main.py around line 3051) and what check_list_maker splices in as the
    # second long block; the case-study notebooks never did it, which is why their
    # "Long 2" columns were a duplicate of "Long 1" rather than a separate arm.
    check_index_first_long = find_best_index(
        dff_test, 1, "long_only_net", name_mode, alpha_range, gamma_range
    )
    check_index_second_long = find_best_index(
        dff_test, 2, "long_only_net", name_mode, alpha_range, gamma_range
    )
    check_index_third_long, best_list_long = find_best_index(
        dff_test, 3, "long_only_net", name_mode, alpha_range, gamma_range
    )
    check_index_fourth_long = find_best_index(
        dff_test, 4, "long_only_net", name_mode, alpha_range, gamma_range
    )  # Genetic algorithm

    base_histogram = dff_test[dff_test["his"] == f"{name_mode} a: 0"].index[0]

    # the base MACD row is the same config under either rule, so it is its own long-only arm
    long_indexes = [
        check_index_first_long,
        check_index_second_long,
        check_index_third_long,
        check_index_fourth_long,
        base_histogram,
    ]

    # print(check_index_first, check_index_second, check_index_third, check_index_fourth, base_histogram)

    dff_valid, current_bet_list_optim = naked_crossover(
        df_valid,
        target_list,
        signal_line_list,
        trade_cost,
        balance,
        risk_per_trade=risk_per_trade,
        signal_line_if=signal_line_if,
        signal_line_if_target=signal_line_if_target,
        last_close=last_close,
    )

    # stash every config's per-trade bets on the validation window, plus which configs
    # the four rules and the base picked, so the significance tests can be run afterwards
    # without re-running the backtest
    validation_bets.append(
        {
            "name_mode": name_mode,
            "signal_line_if": signal_line_if,
            "signal_line_if_target": signal_line_if_target,
            "chosen": {
                "rule_1": dff_valid.iloc[check_index_first]["his"],
                "rule_2": dff_valid.iloc[check_index_second]["his"],
                "rule_3": dff_valid.iloc[check_index_third]["his"],
                "rule_4": dff_valid.iloc[check_index_fourth]["his"],
                "base": dff_valid.iloc[base_histogram]["his"],
            },
            "bets": current_bet_list_optim,
        }
    )

    return (
        dff_valid,
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
        long_indexes,
    )


def extend_with_long_only(output_list, value_df, long_indexes):
    """
    extend_with_long_only appends the genuine long-only arm to each row, taking the long-only figures from the config the LONG-ONLY rule selected rather than from the config the long/short rule selected

    This replaces `row.extend(row[-3:])`, which simply duplicated the three long-only
    columns already on the row. That made the reported "Long 2" identical to "Long 1" by
    construction, so there was no long-only arm at all -- only the long leg of the
    long/short strategy, relabelled.

    Arguments:
        output_list -- the five result rows being assembled (first/second/third/fourth/base)
        value_df -- the validation results frame, whose columns 11:14 are long_only_net, long_only_bets and long_only_wins
        long_indexes -- the five row indexes chosen on "long_only_net", from get_crossover_results

    Returns:
        None -- extends each row of output_list in place
    """
    for position, long_index in enumerate(long_indexes):
        output_list[position].extend(list(value_df.iloc[long_index])[11:14])


def add_df_to_list_detailed(
    output_list,
    value_df,
    index_first,
    index_second,
    index_third,
    index_fourth,
    base_his,
):
    length_of_list = len(value_df.iloc[index_first]) - 1
    index_first_list = list(value_df.iloc[index_first])
    index_second_list = list(value_df.iloc[index_second])
    index_third_list = list(value_df.iloc[index_third])
    index_fourth_list = list(value_df.iloc[index_fourth])
    index_base_list = list(value_df.iloc[base_his])
    for i in range(length_of_list):
        output_list[0][i] += index_first_list[i]
        output_list[1][i] += index_second_list[i]
        output_list[2][i] += index_third_list[i]
        output_list[3][i] += index_fourth_list[i]
        output_list[4][i] += index_base_list[i]


def add_df_to_list_overall(output_list, value_df, index_first, base_his):
    length_of_list = len(value_df.iloc[index_first]) - 1
    index_first_list = list(value_df.iloc[index_first])
    index_base_list = list(value_df.iloc[base_his])
    for i in range(length_of_list):
        output_list[0][i] += index_first_list[i]
        output_list[1][i] += index_base_list[i]


def get_data_rule(rule_data, len_return_list):
    ticker_length = 1
    return_list = [[] for i in range(len_return_list)]

    [
        "net",  # 0
        "netbet",  # 1
        "netpercent",  # 2
        "bets",  # 3
        "wins",  # 4
        "longs",  # 5
        "long_wins",  # 6
        "long_profit",  # 7
        "shorts",  # 8
        "short_wins",  # 9
        "short_profit",  # 10
        "long_only_net",  # 11
        "long_only_bets",  # 12
        "long_only_wins",  # 13
        "long_only_net_other",  # 14
        "long_only_bets_other",  # 15
        "long_only_wins_other",  # 16
        "his",  # 17
    ]
    # [0'net' , 1'netbet' , 2'netpercent' , 3'bets' , 4'wins' , 5'longs' , 6'long_wins' , 7'long_profit' , 8'shorts' , 9'short_wins' , 10'short_profit',
    # 11"long_only_net", 12"long_only_bets", 13"long_only_wins", 14"long_only_net_other", 15"long_only_bets_other", 16"long_only_wins_other", 17'his']

    # avg return for ticker , bets , win/loss , avg long return for ticker , longs , long win ratio , avg short return , short win ratio,
    # 0 / tickers , 3 , 4/3 * % , 7 / tickers , 5 , 6 / 5 * % , 10 / tickers , 9 / 8 * %

    # average long only return for ticker, long only bets, long only win ratio, average other long return for ticker, other long bets, other long win ratio
    # 11 / tickers, 12, 13 / 12 * %, 14 / ticker, 15, 16 / 15 * %
    # print(len(obs_mode_data))
    # print(len(rule_data))
    for i in range(len(rule_data)):
        return_list[i].extend(
            [
                (rule_data[i][0]),  # / ticker_length)
                rule_data[i][3],
                (
                    (rule_data[i][4] * 100 / rule_data[i][3])
                    if rule_data[i][3] != 0
                    else 0
                ),
                (rule_data[i][7]),  # / ticker_length)
                rule_data[i][5],
                (
                    (rule_data[i][6] * 100 / rule_data[i][5])
                    if rule_data[i][5] != 0
                    else 0
                ),
                (rule_data[i][10]),  # / ticker_length)
                (
                    (rule_data[i][9] * 100 / rule_data[i][8])
                    if rule_data[i][8] != 0
                    else 0
                ),
                # Long 1 = the long leg of the config the LONG/SHORT rule picked
                (rule_data[i][11]),  # / ticker_length)
                rule_data[i][12],
                (
                    (rule_data[i][13] * 100 / rule_data[i][12])
                    if rule_data[i][12] != 0
                    else 0
                ),
                # Long 2 = the genuine long-only arm: the config the LONG-ONLY rule picked
                (rule_data[i][14]),  # / ticker_length)
                rule_data[i][15],
                (
                    (rule_data[i][16] * 100 / rule_data[i][15])
                    if rule_data[i][15] != 0
                    else 0
                ),
            ]
        )

    return return_list


def get_best_results(sys_crossover_list):
    best_results_index = 0
    for i in range(1, len(sys_crossover_list) - 1):
        # print(sys_crossover_list[i][0], sys_crossover_list[i-1][0])
        if sys_crossover_list[i][0] > sys_crossover_list[best_results_index][0]:
            best_results_index = i
    return sys_crossover_list[best_results_index]


def table_data_generator(data, indexes, len_return_list):
    output_data = []
    for d in data:
        current_return_list = get_data_rule(d, len_return_list)
        output_data.extend(current_return_list)

    # naked_cols = ['net', 'netbet', 'netpercent', 'bets', 'wins', 'longs', 'long_wins', 'long_profit',
    # 'shorts', 'short_wins', 'short_profit', "long_only_net", "long_only_bets", "long_only_wins",
    # "long_only_net_other", "long_only_bets_other", "long_only_wins_other", 'his']

    tempdf = pd.DataFrame(
        output_data,
        columns=[
            "Return",
            "Transactions",
            "Win Ratio",
            "Long Return",
            "Long Transactions",
            "Long Win Ratio",
            "Short Return",
            "Short Win Ratio",
            "Long 1 Return",
            "Long 1 Transactions",
            "Long 1 Win Ratio",
            "Long 2 Return",
            "Long 2 Transactions",
            "Long 2 Win Ratio",
        ],
        index=indexes,
    )

    return tempdf


alpha_range = [
    -0.9,
    -0.8,
    -0.7,
    -0.6,
    -0.5,
    -0.4,
    -0.3,
    -0.2,
    -0.1,
    0,
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8,
    0.9,
]
gamma_range = [
    1,
    2,
    4,
    10,
    20,
    30,
    40,
    50,
    60,
    70,
    80,
    90,
    100,
    200,
    300,
    400,
    500,
    600,
    700,
    800,
    900,
    1000,
    10000,
    100000,
]
histogram_list, macd_list, signal_line_list = return_macd_lists(
    alpha_range, gamma_range
)

In [ ]:
prefix_tex_tables = "/media/calavera/1TB HDD/Projects/MACD Project/Paper/paper_ready/table_image/"

market = "soybean"
prefix_text = f"./results/{market}"
file_name = f"insert_text_{market}.txt"
time_format = "1_day"

macd_params_basic = (12, 26, 9)
macd_params = macd_params_basic

# macd_heuristics_params = (11.017, 32.925, 16.467)
# macd_heuristics_params = (11, 33, 16)
# macd_params = macd_heuristics_params

# macd_crossover_heuristics_params = (6.866, 33.812, 4.575)
# macd_crossover_heuristics_params = (7, 34, 5)
# macd_params = macd_crossover_heuristics_params

mode = 'a'

params_str = "_".join(str(val) for val in macd_params)
short_period, long_period, signal_period = macd_params

# every (parameters, window) combination writes into its own results folder, and the
# 11-config sweep is about to add eleven more per market, so create the folder instead of
# requiring it to already exist. `from os import listdir` in cell 0 does not bind `os`.
import os

os.makedirs(f"./results/{market}/{params_str}", exist_ok=True)

trade_cost = False
balance = 0
risk_per_trade = 100 / 100
last_close = True

prefix = f"./data/{market}/"

# Wiles traded the single CME August-2014 soybean contract (SQ2014), not a continuous
# series, so that is what his numbers have to be reproduced on.
#
# This replaces soy_data.csv, which is retired. That file is a naively spliced vendor
# "nearby" series and is wrong for this comparison three ways: only 44.3% of its prices sit
# on the CME quarter-cent tick grid (so it is not raw CME settlement data at all), it
# carries about 196 cents of phantom roll P&L inside Wiles's own test window, and its
# buy-and-hold over that window is -221.37 against the -25.3 he reports. Our old soybean
# "profit" came from those roll gaps rather than from the strategy.
#
# This file reproduces his buy-and-hold as -25.25, and is 100% on the tick grid. It is
# comma separated with an ISO date, where soy_data.csv was semicolon separated.
soybean_data = pd.read_csv(
    prefix + "CME_SQ2014.csv",
    index_col="Date",
    parse_dates=True,
    sep=",",
    usecols=["Date", "Close", "Open", "High", "Low"],
)

soybean_data.dropna(inplace=True)
soybean_data.columns = list(map(lambda x: x.lower(), list(soybean_data.columns)))
soybean_data.sort_index(inplace=True)
soybean_data.index.names = ["date"]

soybean_data = calculate_indicators(
    soybean_data,
    col="close",
    ema_mode="normal",
    short_period=short_period,
    long_period=long_period,
    signal_period=signal_period,
    alpha_range=alpha_range,
    gamma_range=gamma_range,
)

# the engine's strategies execute at a 'fill' column; without it macd_crossover raises KeyError.
# on daily bars every gap exceeds the session threshold, so fill == close, i.e. same-bar
# execution -- the same convention these published results were produced under.
add_fill_column(soybean_data)

soybean_data

,open,high,low,close,MACD,SignalLine,Histogram,MACD a: -0.9,MACD Signal a: -0.9,Histogram a: -0.9,...,Histogram a: 0.9 g: 1000,MACD a: 0.9 g: 1000 times,MACD a: 0.9 g: 10000,MACD Signal a: 0.9 g: 10000,Histogram a: 0.9 g: 10000,MACD a: 0.9 g: 10000 times,MACD a: 0.9 g: 100000,MACD Signal a: 0.9 g: 100000,Histogram a: 0.9 g: 100000,MACD a: 0.9 g: 100000 times
2011-02-25,1339.75,1398.50,1338.25,1371.75,-31.686730,-27.676122,-4.010609,-17.356902,-28.285971,10.929069,...,-3.626854,969,-16.716014,-13.089160,-3.626854,972,-16.716014,-13.089160,-3.626854,973
2011-02-28,1364.00,1368.50,1358.00,1365.75,-28.508936,-27.953727,-0.555209,-38.545766,-31.705903,-6.839863,...,-3.415950,969,-18.213085,-14.797135,-3.415950,972,-18.213085,-14.797135,-3.415950,973
2011-03-01,1372.50,1375.00,1353.75,1372.75,-24.504511,-26.803988,2.299477,-11.867381,-25.093062,13.225681,...,-2.963690,969,-19.242670,-16.278980,-2.963690,972,-19.242670,-16.278980,-2.963690,973
2011-03-02,1395.75,1399.00,1375.00,1392.75,-17.465603,-23.691193,6.225590,-22.504003,-24.230042,1.726040,...,-2.326583,969,-19.768854,-17.442272,-2.326583,972,-19.768854,-17.442272,-2.326583,973
2011-03-03,1404.50,1414.25,1402.50,1410.00,-8.911795,-18.764727,9.852932,3.321192,-15.046298,18.367489,...,-1.397505,969,-19.538529,-18.141024,-1.397505,972,-19.538529,-18.141024,-1.397505,973
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2014-12-24,1036.12,1041.38,1025.62,1028.62,6.333834,7.579886,-1.246052,4.515589,6.993106,-2.477517,...,-0.490494,969,9.919641,10.410134,-0.490494,972,9.919641,10.410134,-0.490494,973
2014-12-26,1032.12,1047.88,1030.88,1047.12,8.707360,7.955711,0.751650,12.479954,8.822055,3.657899,...,-0.566050,969,9.561060,10.127110,-0.566050,972,9.561060,10.127110,-0.566050,973
2014-12-29,1053.50,1067.75,1046.12,1049.62,10.743535,8.884985,1.858549,9.180757,8.941623,0.239134,...,-0.434280,969,9.475690,9.909970,-0.434280,972,9.475690,9.909970,-0.434280,973
2014-12-30,1048.62,1055.88,1042.38,1042.62,10.679922,9.483297,1.196624,12.029170,9.970805,2.058365,...,-0.204997,969,9.602474,9.807471,-0.204997,972,9.602474,9.807471,-0.204997,973


In [3]:
start_optimize_soybean = pd.Timestamp("2012-01-03")
end_optimize_soybean = pd.Timestamp("2013-12-24")
start_test_soybean = pd.Timestamp("2013-12-26")
end_test_soybean = pd.Timestamp("2014-08-14")

optimize_data = soybean_data[start_optimize_soybean:end_optimize_soybean]
test_data = soybean_data[start_test_soybean:end_test_soybean]

# optimize_data
test_data

,open,high,low,close,MACD,SignalLine,Histogram,MACD a: -0.9,MACD Signal a: -0.9,Histogram a: -0.9,...,Histogram a: 0.9 g: 1000,MACD a: 0.9 g: 1000 times,MACD a: 0.9 g: 10000,MACD Signal a: 0.9 g: 10000,Histogram a: 0.9 g: 10000,MACD a: 0.9 g: 10000 times,MACD a: 0.9 g: 100000,MACD Signal a: 0.9 g: 100000,Histogram a: 0.9 g: 100000,MACD a: 0.9 g: 100000 times
2013-12-26,1329.50,1329.50,1316.75,1319.75,9.079610,12.271635,-3.192025,8.318849,11.514719,-3.195870,...,0.072025,969,13.615873,13.543936,0.071937,972,13.615873,13.543937,0.071936,973
2013-12-27,1318.75,1332.38,1318.75,1331.75,9.055766,11.199679,-2.143913,9.718992,10.916143,-1.197151,...,-0.254401,969,13.162247,13.416707,-0.254460,972,13.162247,13.416707,-0.254460,973
2013-12-30,1313.60,1315.88,1299.88,1308.88,4.498894,8.966084,-4.467190,-0.199194,7.211031,-7.410225,...,-0.443366,969,12.751599,13.195004,-0.443405,972,12.751599,13.195004,-0.443406,973
2013-12-31,1307.38,1310.88,1289.38,1289.38,-2.548450,5.127906,-7.676356,-4.662780,3.253094,-7.915874,...,-0.845758,969,11.926328,12.772112,-0.845784,972,11.926328,12.772112,-0.845784,973
2014-01-02,1288.88,1288.88,1262.88,1270.75,-11.088451,-0.277546,-10.810905,-16.871554,-3.455122,-13.416432,...,-1.528824,969,10.478850,12.007691,-1.528841,972,10.478850,12.007692,-1.528841,973
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2014-08-08,1098.00,1114.12,1097.38,1113.88,-98.752586,-107.025345,8.272759,-86.609689,-103.115964,16.506275,...,-2.126116,969,-103.720319,-101.594204,-2.126116,972,-103.720319,-101.594204,-2.126116,973
2014-08-11,1111.75,1128.62,1105.88,1108.00,-92.673890,-102.241527,9.567636,-98.131671,-101.454533,3.322862,...,-1.086228,969,-103.223546,-102.137318,-1.086228,972,-103.223546,-102.137318,-1.086228,973
2014-08-12,1109.00,1114.25,1079.88,1094.75,-89.606828,-98.029961,8.423132,-81.934470,-94.947845,13.013376,...,-0.020842,969,-102.168581,-102.147739,-0.020842,972,-102.168581,-102.147739,-0.020842,973
2014-08-13,1093.50,1102.62,1078.88,1079.50,-89.098951,-95.052958,5.954006,-95.546985,-95.147558,-0.399426,...,0.823556,969,-100.912405,-101.735961,0.823556,972,-100.912405,-101.735961,0.823556,973


SYSTEM 1 - NO EXTRA CONDITION

In [4]:
signal_line_if = False
signal_line_if_target = False

Signalline crossover

In [5]:
sys1_major_macd_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "MACDCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    long_indexes,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    histogram_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys1_major_macd_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
# the long-only arm comes from the config the long-only rule chose, not from a
# copy of this row's own long columns
extend_with_long_only(sys1_major_macd_crossover, dff, long_indexes)
for i in range(len(sys1_major_macd_crossover)):
    print(sys1_major_macd_crossover[i], len(sys1_major_macd_crossover[i]))

best_sys1_major_macd_crossover = get_best_results(sys1_major_macd_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(27.650000000000546), np.float64(1.2568181818182067), np.int64(0), np.int64(22), np.int64(4), np.int64(11), np.int64(2), np.float64(-85.23999999999978), np.int64(11), np.int64(2), np.float64(112.89000000000033), np.float64(-85.23999999999978), np.int64(11), np.int64(2), np.float64(-85.23999999999978), np.int64(11), np.int64(2)] 17
[np.float64(253.65000000000055), np.float64(9.755769230769252), np.int64(0), np.int64(26), np.int64(7), np.int64(13), np.int64(3), np.float64(27.76000000000022), np.int64(13), np.int64(4), np.float64(225.89000000000033), np.float64(27.76000000000022), np.int64(13), np.int64(3), np.float64(27.76000000000022), np.int64(13), np.int64(3)] 17
[np.float64(253.65000000000055), np.float64(9.755769230769252), np.int64(0), np.int64(26), np.int64(7), np.int64(13), np.int64(3), np.float64(27.76000000000022), np.int64(13), np.int64(4), np.float64(225.89000000000033), np.float64(27.76000000000022), np.int64(13), np.int64(3), np.float64(27.76000000000022), np.int

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
6,27.65,1.256818,0,22,4,11,2,-85.24,11,2,112.89,-85.24,11,2,Histogram a: -0.3
3,253.65,9.755769,0,26,7,13,3,27.76,13,4,225.89,27.76,13,3,Histogram a: -0.6
3,253.65,9.755769,0,26,7,13,3,27.76,13,4,225.89,27.76,13,3,Histogram a: -0.6
3,253.65,9.755769,0,26,7,13,3,27.76,13,4,225.89,27.76,13,3,Histogram a: -0.6
9,29.87,1.659444,0,18,4,9,2,-84.13,9,2,114.00,-84.13,9,2,Histogram a: 0


Zeroline crossover

In [6]:
sys1_major_zero_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "ZEROCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    long_indexes,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    macd_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys1_major_zero_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
# the long-only arm comes from the config the long-only rule chose, not from a
# copy of this row's own long columns
extend_with_long_only(sys1_major_zero_crossover, dff, long_indexes)
for i in range(len(sys1_major_zero_crossover)):
    print(sys1_major_zero_crossover[i], len(sys1_major_zero_crossover[i]))

best_sys1_major_zero_crossover = get_best_results(sys1_major_zero_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(43.0), np.float64(10.75), np.int64(0), np.int64(4), np.int64(1), np.int64(2), np.int64(1), np.float64(105.61999999999989), np.int64(2), np.int64(0), np.float64(-62.61999999999989), np.float64(105.61999999999989), np.int64(2), np.int64(1), np.float64(105.61999999999989), np.int64(2), np.int64(1)] 17
[np.float64(46.5), np.float64(23.25), np.int64(0), np.int64(2), np.int64(1), np.int64(1), np.int64(1), np.float64(99.5), np.int64(1), np.int64(0), np.float64(-53.0), np.float64(99.5), np.int64(1), np.int64(1), np.float64(99.5), np.int64(1), np.int64(1)] 17
[np.float64(27.38000000000011), np.float64(13.690000000000055), np.int64(0), np.int64(2), np.int64(1), np.int64(1), np.int64(1), np.float64(89.38000000000011), np.int64(1), np.int64(0), np.float64(-62.0), np.float64(89.38000000000011), np.int64(1), np.int64(1), np.float64(89.38000000000011), np.int64(1), np.int64(1)] 17
[np.float64(27.38000000000011), np.float64(13.690000000000055), np.int64(0), np.int64(2), np.int64(1), np.int

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
7,43.00,10.75,0,4,1,2,1,105.62,2,0,-62.62,105.62,2,1,MACD a: -0.2
18,46.50,23.25,0,2,1,1,1,99.50,1,0,-53.00,99.50,1,1,MACD a: 0.9
427,27.38,13.69,0,2,1,1,1,89.38,1,0,-62.00,89.38,1,1,MACD a: 0.8 g: 1
438,27.38,13.69,0,2,1,1,1,89.38,1,0,-62.00,89.38,1,1,MACD a: 0.8 g: 90
9,115.24,57.62,0,2,1,1,1,141.74,1,0,-26.50,141.74,1,1,MACD a: 0


SYSTEM 2 - SIGNAL EXTRA CONDITION

In [7]:
signal_line_if = True
signal_line_if_target = False

Signalline crossover

In [8]:
sys2_major_macd_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "MACDCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    long_indexes,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    histogram_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys2_major_macd_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
# the long-only arm comes from the config the long-only rule chose, not from a
# copy of this row's own long columns
extend_with_long_only(sys2_major_macd_crossover, dff, long_indexes)
for i in range(len(sys2_major_macd_crossover)):
    print(sys2_major_macd_crossover[i], len(sys2_major_macd_crossover[i]))

best_sys2_major_macd_crossover = get_best_results(sys2_major_macd_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(365.0), np.float64(182.5), np.int64(0), np.int64(2), np.int64(2), np.int64(1), np.int64(1), np.float64(123.75), np.int64(1), np.int64(1), np.float64(241.25), np.float64(123.75), np.int64(1), np.int64(1), np.float64(123.75), np.int64(1), np.int64(1)] 17
[np.float64(365.0), np.float64(182.5), np.int64(0), np.int64(2), np.int64(2), np.int64(1), np.int64(1), np.float64(123.75), np.int64(1), np.int64(1), np.float64(241.25), np.float64(123.75), np.int64(1), np.int64(1), np.float64(123.75), np.int64(1), np.int64(1)] 17
[np.float64(365.0), np.float64(182.5), np.int64(0), np.int64(2), np.int64(2), np.int64(1), np.int64(1), np.float64(123.75), np.int64(1), np.int64(1), np.float64(241.25), np.float64(123.75), np.int64(1), np.int64(1), np.float64(123.75), np.int64(1), np.int64(1)] 17
[np.float64(365.0), np.float64(182.5), np.int64(0), np.int64(2), np.int64(2), np.int64(1), np.int64(1), np.float64(123.75), np.int64(1), np.int64(1), np.float64(241.25), np.float64(123.75), np.int64(1), np

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
4,365.00,182.500,0,2,2,1,1,123.75,1,1,241.25,123.75,1,1,Histogram a: -0.5
4,365.00,182.500,0,2,2,1,1,123.75,1,1,241.25,123.75,1,1,Histogram a: -0.5
4,365.00,182.500,0,2,2,1,1,123.75,1,1,241.25,123.75,1,1,Histogram a: -0.5
4,365.00,182.500,0,2,2,1,1,123.75,1,1,241.25,123.75,1,1,Histogram a: -0.5
9,345.49,172.745,0,2,2,1,1,117.37,1,1,228.12,117.37,1,1,Histogram a: 0


Zeroline crossover

In [9]:
sys2_major_zero_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "ZEROCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    long_indexes,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    macd_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys2_major_zero_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
# the long-only arm comes from the config the long-only rule chose, not from a
# copy of this row's own long columns
extend_with_long_only(sys2_major_zero_crossover, dff, long_indexes)
for i in range(len(sys2_major_zero_crossover)):
    print(sys2_major_zero_crossover[i], len(sys2_major_zero_crossover[i]))

best_sys2_major_zero_crossover = get_best_results(sys2_major_zero_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(95.0), np.float64(47.5), np.int64(0), np.int64(2), np.int64(1), np.int64(1), np.int64(1), np.float64(131.6199999999999), np.int64(1), np.int64(0), np.float64(-36.61999999999989), np.float64(131.6199999999999), np.int64(1), np.int64(1), np.float64(131.6199999999999), np.int64(1), np.int64(1)] 17
[np.float64(197.3800000000001), np.float64(98.69000000000005), np.int64(0), np.int64(2), np.int64(2), np.int64(1), np.int64(1), np.float64(188.5), np.int64(1), np.int64(1), np.float64(8.88000000000011), np.float64(188.5), np.int64(1), np.int64(1), np.float64(188.5), np.int64(1), np.int64(1)] 17
[np.float64(0.0), np.float64(0.0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.float64(0.0), np.int64(0), np.int64(0), np.float64(0.0), np.float64(0.0), np.int64(0), np.int64(0), np.float64(0.0), np.int64(0), np.int64(0)] 17
[np.float64(0.0), np.float64(0.0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.float64(0.0), np.int64(0), np.int64(0), np.f

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
11,95.00,47.50,0,2,1,1,1,131.62,1,0,-36.62,131.62,1,1,MACD a: 0.2
19,197.38,98.69,0,2,2,1,1,188.50,1,1,8.88,188.50,1,1,MACD a: -0.9 g: 1
116,0.00,0.00,0,0,0,0,0,0.00,0,0,0.00,0.00,0,0,MACD a: -0.5 g: 2
127,0.00,0.00,0,0,0,0,0,0.00,0,0,0.00,0.00,0,0,MACD a: -0.5 g: 100
9,115.24,57.62,0,2,1,1,1,141.74,1,0,-26.50,141.74,1,1,MACD a: 0


SYSTEM 3 - Target SIGNAL EXTRA CONDITION

In [10]:
signal_line_if = False
signal_line_if_target = True

Signalline crossover

In [11]:
sys3_major_macd_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "MACDCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    long_indexes,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    histogram_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys3_major_macd_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
# the long-only arm comes from the config the long-only rule chose, not from a
# copy of this row's own long columns
extend_with_long_only(sys3_major_macd_crossover, dff, long_indexes)
for i in range(len(sys3_major_macd_crossover)):
    print(sys3_major_macd_crossover[i], len(sys3_major_macd_crossover[i]))

best_sys3_major_macd_crossover = get_best_results(sys3_major_macd_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(365.0), np.float64(182.5), np.int64(0), np.int64(2), np.int64(2), np.int64(1), np.int64(1), np.float64(123.75), np.int64(1), np.int64(1), np.float64(241.25), np.float64(123.75), np.int64(1), np.int64(1), np.float64(123.75), np.int64(1), np.int64(1)] 17
[np.float64(365.0), np.float64(182.5), np.int64(0), np.int64(2), np.int64(2), np.int64(1), np.int64(1), np.float64(123.75), np.int64(1), np.int64(1), np.float64(241.25), np.float64(123.75), np.int64(1), np.int64(1), np.float64(123.75), np.int64(1), np.int64(1)] 17
[np.float64(365.0), np.float64(182.5), np.int64(0), np.int64(2), np.int64(2), np.int64(1), np.int64(1), np.float64(123.75), np.int64(1), np.int64(1), np.float64(241.25), np.float64(123.75), np.int64(1), np.int64(1), np.float64(123.75), np.int64(1), np.int64(1)] 17
[np.float64(365.0), np.float64(182.5), np.int64(0), np.int64(2), np.int64(2), np.int64(1), np.int64(1), np.float64(123.75), np.int64(1), np.int64(1), np.float64(241.25), np.float64(123.75), np.int64(1), np

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
4,365.00,182.500,0,2,2,1,1,123.75,1,1,241.25,123.75,1,1,Histogram a: -0.5
4,365.00,182.500,0,2,2,1,1,123.75,1,1,241.25,123.75,1,1,Histogram a: -0.5
4,365.00,182.500,0,2,2,1,1,123.75,1,1,241.25,123.75,1,1,Histogram a: -0.5
3,365.00,182.500,0,2,2,1,1,123.75,1,1,241.25,123.75,1,1,Histogram a: -0.6
9,345.49,172.745,0,2,2,1,1,117.37,1,1,228.12,117.37,1,1,Histogram a: 0


Zerocross crossover

In [12]:
sys3_major_zero_crossover = [
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
    [0 for i in range(14)],
]
strategy = "ZEROCROSS"
name_mode = get_name_mode(strategy)
(
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
    long_indexes,
) = get_crossover_results(
    optimize_data,
    test_data,
    name_mode,
    macd_list,
    signal_line_list,
    trade_cost,
    balance,
    risk_per_trade,
    alpha_range,
    gamma_range,
    signal_line_if,
    signal_line_if_target,
    last_close,
)

add_df_to_list_detailed(
    sys3_major_zero_crossover,
    dff,
    check_index_first,
    check_index_second,
    check_index_third,
    check_index_fourth,
    base_histogram,
)
# the long-only arm comes from the config the long-only rule chose, not from a
# copy of this row's own long columns
extend_with_long_only(sys3_major_zero_crossover, dff, long_indexes)
for i in range(len(sys3_major_zero_crossover)):
    print(sys3_major_zero_crossover[i], len(sys3_major_zero_crossover[i]))

best_sys3_major_zero_crossover = get_best_results(sys3_major_zero_crossover)

dff.iloc[
    [
        check_index_first,
        check_index_second,
        check_index_third,
        check_index_fourth,
        base_histogram,
    ]
]

[np.float64(64.76000000000022), np.float64(32.38000000000011), np.int64(0), np.int64(2), np.int64(1), np.int64(1), np.int64(1), np.float64(125.13000000000011), np.int64(1), np.int64(0), np.float64(-60.36999999999989), np.float64(125.13000000000011), np.int64(1), np.int64(1), np.float64(125.13000000000011), np.int64(1), np.int64(1)] 17
[np.float64(117.14000000000033), np.float64(58.570000000000164), np.int64(0), np.int64(2), np.int64(1), np.int64(1), np.int64(1), np.float64(144.3800000000001), np.int64(1), np.int64(0), np.float64(-27.23999999999978), np.float64(144.3800000000001), np.int64(1), np.int64(1), np.float64(144.3800000000001), np.int64(1), np.int64(1)] 17
[np.float64(117.14000000000033), np.float64(58.570000000000164), np.int64(0), np.int64(2), np.int64(1), np.int64(1), np.int64(1), np.float64(144.3800000000001), np.int64(1), np.int64(0), np.float64(-27.23999999999978), np.float64(144.3800000000001), np.int64(1), np.int64(1), np.float64(144.3800000000001), np.int64(1), np.int6

,net,netbet,netpercent,bets,wins,longs,long_wins,long_profit,shorts,short_wins,short_profit,long_only_net,long_only_bets,long_only_wins,his
284,64.76,32.38,0,2,1,1,1,125.13,1,0,-60.37,125.13,1,1,MACD a: 0.2 g: 2
2,117.14,58.57,0,2,1,1,1,144.38,1,0,-27.24,144.38,1,1,MACD a: -0.7
2,117.14,58.57,0,2,1,1,1,144.38,1,0,-27.24,144.38,1,1,MACD a: -0.7
448,27.38,13.69,0,2,1,1,1,89.38,1,0,-62.00,89.38,1,1,MACD a: 0.8 g: 1000
9,115.24,57.62,0,2,1,1,1,141.74,1,0,-26.50,141.74,1,1,MACD a: 0


Getting Tables Detailed

Signalline crossover

In [13]:
# sys_modes = ["System 1", "System 2", "System 3"]
# # sys_modes = ["System 1", "System 2"]
# entries = ["Rule 1", "Rule 2", "Rule 3", "GA", "MACD"]
# tuples = []

# data = [sys1_major_macd_crossover, sys2_major_macd_crossover, sys3_major_macd_crossover]
# # data = [sys1_major_macd_crossover, sys2_major_macd_crossover]

# for sys_mode in sys_modes:
#     for entry in entries:
#         tuples.append((sys_mode, entry))
# table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

# table_data = table_data_generator(data, table_index, 5)

# columns = [
#     "Return",
#     "Win Ratio",
#     "Long Return",
#     "Long Win Ratio",
#     "Short Return",
#     "Short Win Ratio",
#     # "Long 1 Return",
#     # "Long 1 Win Ratio",
#     # "Long 2 Return",
#     # "Long 2 Win Ratio",
# ]
# for col in columns:
#     l = []
#     for x in table_data[col]:

#         if col[-6:] == "Return":
#             l.append(round(100 * x))
#         else:
#             l.append(round(x, 2))
#     table_data[col] = l

# table_data.to_csv(f"./results/soybean/{params_str}/soy_rules_macdcross.csv", index=True)

# table_data

In [14]:
# # sys_modes = ["System 1", "System 2", "System 3"]
# sys_modes = ["System 1", "System 2"]
# entries = ["Best", "MACD"]
# tuples = []

# # data = [[best_sys1_major_macd_crossover, sys1_major_macd_crossover[-1]], [best_sys2_major_macd_crossover, sys2_major_macd_crossover[-1]], [best_sys3_major_macd_crossover, sys3_major_macd_crossover[-1]]]
# data = [
#     [best_sys1_major_macd_crossover, sys1_major_macd_crossover[-1]],
#     [best_sys2_major_macd_crossover, sys2_major_macd_crossover[-1]],
# ]

# for sys_mode in sys_modes:
#     for entry in entries:
#         tuples.append((sys_mode, entry))
# table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

# table_data = table_data_generator(data, table_index, 2)

# columns = [
#     "Return",
#     "Win Ratio",
#     "Long Return",
#     "Long Win Ratio",
#     "Short Return",
#     "Short Win Ratio",
#     # "Long 1 Return",
#     # "Long 1 Win Ratio",
#     # "Long 2 Return",
#     # "Long 2 Win Ratio",
# ]
# for col in columns:
#     l = []
#     for x in table_data[col]:
#         if col[-6:] == "Return":
#             l.append(round(100 * x))
#         else:
#             l.append(round(x, 2))
#     table_data[col] = l

# table_data.to_csv(
#     f"./results/soybean/{params_str}/soy_rules_macdcross_overall.csv", index=True
# )

# table_data

Zeroline crossover

In [15]:
# sys_modes = ["System 1", "System 2", "System 3"]
# # sys_modes = ["System 1", "System 2"]
# entries = ["Rule 1", "Rule 2", "Rule 3", "GA", "MACD"]
# tuples = []

# data = [sys1_major_zero_crossover, sys2_major_zero_crossover, sys3_major_zero_crossover]
# # data = [sys1_major_zero_crossover, sys2_major_zero_crossover]

# for sys_mode in sys_modes:
#     for entry in entries:
#         tuples.append((sys_mode, entry))
# table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

# table_data = table_data_generator(data, table_index, 5)

# columns = [
#     "Return",
#     "Win Ratio",
#     "Long Return",
#     "Long Win Ratio",
#     "Short Return",
#     "Short Win Ratio",
#     # "Long 1 Return",
#     # "Long 1 Win Ratio",
#     # "Long 2 Return",
#     # "Long 2 Win Ratio",
# ]
# for col in columns:
#     l = []
#     for x in table_data[col]:
#         if col[-6:] == "Return":
#             l.append(round(100 * x))
#         else:
#             l.append(round(x, 2))
#     table_data[col] = l

# table_data.to_csv(f"./results/soybean/{params_str}/soy_rules_zerocross.csv", index=True)

# table_data

In [16]:
# # sys_modes = ["System 1", "System 2", "System 3"]
# sys_modes = ["System 1", "System 2"]
# entries = ["Best", "MACD"]
# tuples = []

# # data = [[best_sys1_major_zero_crossover, sys1_major_zero_crossover[-1]], [best_sys2_major_zero_crossover, sys2_major_zero_crossover[-1]], [best_sys3_major_zero_crossover, sys3_major_zero_crossover[-1]]]
# data = [
#     [best_sys1_major_zero_crossover, sys1_major_zero_crossover[-1]],
#     [best_sys2_major_zero_crossover, sys2_major_zero_crossover[-1]],
# ]

# for sys_mode in sys_modes:
#     for entry in entries:
#         tuples.append((sys_mode, entry))
# table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])

# table_data = table_data_generator(data, table_index, 2)

# columns = [
#     "Return",
#     "Win Ratio",
#     "Long Return",
#     "Long Win Ratio",
#     "Short Return",
#     "Short Win Ratio",
#     # "Long 1 Return",
#     # "Long 1 Win Ratio",
#     # "Long 2 Return",
#     # "Long 2 Win Ratio",
# ]
# for col in columns:
#     l = []
#     for x in table_data[col]:
#         if col[-6:] == "Return":
#             l.append(round(100 * x))
#         else:
#             l.append(round(x, 2))
#     table_data[col] = l

# table_data.to_csv(
#     f"./results/soybean/{params_str}/soy_rules_zerocross_overall.csv", index=True
# )

# table_data

In [17]:
crossover_modes = ["MACD Crossover", "Zero Crossover"]
sys_modes = ["System 1", "System 2", "System 3"]
entries = ["Rule 1", "Rule 2", "Rule 3", "GA", "MACD"]
tuples = []

data = [sys1_major_macd_crossover, sys2_major_macd_crossover, sys3_major_macd_crossover,
        sys1_major_zero_crossover, sys2_major_zero_crossover, sys3_major_zero_crossover]
# data = [sys1_major_macd_crossover, sys2_major_macd_crossover]

for crossover_mode in crossover_modes:
    for sys_mode in sys_modes:
        for entry in entries:
            tuples.append((crossover_mode, sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second", "third"])

table_data = table_data_generator(data, table_index, 5)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # long-only arms: "Long 1" = long-only P&L of the config the LONG/SHORT rule picked,
    # "Long 2" = long-only P&L of the config the LONG-ONLY rule picked (the true long-only
    # strategy, and the apples-to-apples arm against any long/flat paper)
    "Long 1 Return",
    "Long 1 Win Ratio",
    "Long 2 Return",
    "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:

        if col[-6:] == "Return":
            # NO x100 here: the engine's net is already cents per bushel (amount = 1 unit,
            # pnl = close - close), which is exactly Wiles's unit. the old x100 inflated our
            # side of the comparison by 100x against his 3906 / 664.5 figures.
            l.append(round(x, 2))
        else:
            l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(f"./results/soybean/{params_str}/soy_rules.csv", index=True)
caption = f"Results for {macd_params} paramters for MACD by applying the normal scenario on \\citet{"{WILES201585}"}'s data"
write_table_tex(table_data, f"./results/soybean/{params_str}", "soy_rules", f"Soybean_{params_str}", caption)
write_table_tex(table_data, prefix_tex_tables, f"soy_rules_{params_str}", f"Soybean_{params_str}", caption)
text = f"\\input{"{"}soy_rules_{params_str}_tex{"}"}"
if macd_params == (12, 26, 9):
    write_to_text(prefix_text, file_name, text, mode='w')
else:
    write_to_text(prefix_text, file_name, text, mode='a')
    
table_data

Return  Transactions  Win Ratio  Long Return  \
first          second   third                                                  
MACD Crossover System 1 Rule 1    2765            22      18.18        -8524   
                        Rule 2   25365            26      26.92         2776   
                        Rule 3   25365            26      26.92         2776   
                        GA       25365            26      26.92         2776   
                        MACD      2987            18      22.22        -8413   
               System 2 Rule 1   36500             2     100.00        12375   
                        Rule 2   36500             2     100.00        12375   
                        Rule 3   36500             2     100.00        12375   
                        GA       36500             2     100.00        12375   
                        MACD     34549             2     100.00        11737   
               System 3 Rule 1   36500             2     100.00        12375   
                        Rule 2   36500             2     100.00        12375   
                        Rule 3   36500             2     100.00        12375   
                        GA       36500             2     100.00        12375   
                        MACD     34549             2     100.00        11737   
Zero Crossover System 1 Rule 1    4300             4      25.00        10562   
                        Rule 2    4650             2      50.00         9950   
                        Rule 3    2738             2      50.00         8938   
                        GA        2738             2      50.00         8938   
                        MACD     11524             2      50.00        14174   
               System 2 Rule 1    9500             2      50.00        13162   
                        Rule 2   19738             2     100.00        18850   
                        Rule 3       0             0       0.00            0   
                        GA           0             0       0.00            0   
                        MACD     11524             2      50.00        14174   
               System 3 Rule 1    6476             2      50.00        12513   
                        Rule 2   11714             2      50.00        14438   
                        Rule 3   11714             2      50.00        14438   
                        GA        2738             2      50.00         8938   
                        MACD     11524             2      50.00        14174   

                                Long Transactions  Long Win Ratio  \
first          second   third                                       
MACD Crossover System 1 Rule 1                 11           18.18   
                        Rule 2                 13           23.08   
                        Rule 3                 13           23.08   
                        GA                     13           23.08   
                        MACD                    9           22.22   
               System 2 Rule 1                  1          100.00   
                        Rule 2                  1          100.00   
                        Rule 3                  1          100.00   
                        GA                      1          100.00   
                        MACD                    1          100.00   
               System 3 Rule 1                  1          100.00   
                        Rule 2                  1          100.00   
                        Rule 3                  1          100.00   
                        GA                      1          100.00   
                        MACD                    1          100.00   
Zero Crossover System 1 Rule 1                  2           50.00   
                        Rule 2                  1          100.00   
                        Rule 3                  1          100.00   
                        GA                      1          100.00   
                      

In [18]:
crossover_modes = ["MACD Crossover", "Zero Crossover"]
sys_modes = ["System 1", "System 2"]
entries = ["Best", "MACD"]
tuples = []

# data = [[best_sys1_major_zero_crossover, sys1_major_zero_crossover[-1]], [best_sys2_major_zero_crossover, sys2_major_zero_crossover[-1]], [best_sys3_major_zero_crossover, sys3_major_zero_crossover[-1]]]
data = [
    [best_sys1_major_macd_crossover, sys1_major_macd_crossover[-1]],
    [best_sys2_major_macd_crossover, sys2_major_macd_crossover[-1]],
    [best_sys1_major_zero_crossover, sys1_major_zero_crossover[-1]],
    [best_sys2_major_zero_crossover, sys2_major_zero_crossover[-1]],
]

for crossover_mode in crossover_modes:
    for sys_mode in sys_modes:
        for entry in entries:
            tuples.append((crossover_mode, sys_mode, entry))
table_index = pd.MultiIndex.from_tuples(tuples, names=["first", "second", "third"])

table_data = table_data_generator(data, table_index, 2)

columns = [
    "Return",
    "Win Ratio",
    "Long Return",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    # long-only arms: "Long 1" = long-only P&L of the config the LONG/SHORT rule picked,
    # "Long 2" = long-only P&L of the config the LONG-ONLY rule picked (the true long-only
    # strategy, and the apples-to-apples arm against any long/flat paper)
    "Long 1 Return",
    "Long 1 Win Ratio",
    "Long 2 Return",
    "Long 2 Win Ratio",
]
for col in columns:
    l = []
    for x in table_data[col]:
        # print(x)
        if col[-6:] == "Return":
            # NO x100 here: the engine's net is already cents per bushel (amount = 1 unit,
            # pnl = close - close), which is exactly Wiles's unit. the old x100 inflated our
            # side of the comparison by 100x against his 3906 / 664.5 figures.
            l.append(round(x, 2))
        else:
            l.append(round(x, 2))
    table_data[col] = l

table_data.to_csv(
    f"./results/soybean/{params_str}/soy_rules_overall.csv", index=True
)


# The csv keeps all fourteen columns. The tex table takes a chosen subset, because
# "Long 1" is the long LEG of the long/short strategy and is structurally the same as
# "Long Return" for an always-in-market crossover system, so printing both would just
# repeat a column. "Long 2" is the separate long-only strategy and is the one worth
# showing. The first eight must stay in this order: the published comparison rows line
# up against them.
tex_columns = [
    "Return",
    "Transactions",
    "Win Ratio",
    "Long Return",
    "Long Transactions",
    "Long Win Ratio",
    "Short Return",
    "Short Win Ratio",
    "Long 2 Return",
    "Long 2 Transactions",
    "Long 2 Win Ratio",
]
tex_data = table_data[tex_columns].copy()

# Wiles reports both a total ("profit in cents per bushel for all trades") and a profit PER
# ROUND TRIP, and the per-round-trip figure is the honest one to compare on: his 3906 is a
# sum over about 46 round trips, so setting it beside a single-trade number would mislead.
# His four are 81.4, 90.1, 132.9 and 145.6. Our Transactions column counts round trips, so
# this is simply the total divided by it.
tex_data["Per Round Trip"] = (
    tex_data["Return"] / tex_data["Transactions"].replace(0, pd.NA)
).astype(float)

if macd_params == (12, 26, 9):
    caption = "Comparison between the results published in \\citet{WILES201585} and the results obtained by applying the normal scenario on their data"
    write_overall_table_tex(tex_data, f"./results/soybean/{params_str}", "soy_rules_overall", f"Soybean_{params_str}_overall", caption, "wiles")
    write_overall_table_tex(tex_data, prefix_tex_tables, f"soy_rules_overall_{params_str}", f"Soybean_{params_str}_overall", caption, "wiles")
else:
    write_overall_table_tex(tex_data, f"./results/soybean/{params_str}", "soy_rules_overall", f"Soybean_{params_str}_overall", caption, None)
    write_overall_table_tex(tex_data, prefix_tex_tables, f"soy_rules_overall_{params_str}", f"Soybean_{params_str}_overall", caption, None)
text = f"\\input{"{"}soy_rules_overall_{params_str}_tex{"}"}"
write_to_text(prefix_text, file_name, text, mode='a')
    
table_data

Return  Transactions  Win Ratio  Long Return  \
first          second   third                                                 
MACD Crossover System 1 Best    25365            26      26.92         2776   
                        MACD     2987            18      22.22        -8413   
               System 2 Best    36500             2     100.00        12375   
                        MACD    34549             2     100.00        11737   
Zero Crossover System 1 Best     4650             2      50.00         9950   
                        MACD    11524             2      50.00        14174   
               System 2 Best    19738             2     100.00        18850   
                        MACD    11524             2      50.00        14174   

                               Long Transactions  Long Win Ratio  \
first          second   third                                      
MACD Crossover System 1 Best                  13           23.08   
                        MACD                   9           22.22   
               System 2 Best                   1          100.00   
                        MACD                   1          100.00   
Zero Crossover System 1 Best                   1          100.00   
                        MACD                   1          100.00   
               System 2 Best                   1          100.00   
                        MACD                   1          100.00   

                               Short Return  Short Win Ratio  
first          second   third                                 
MACD Crossover System 1 Best          22589            30.77  
                        MACD          11400            22.22  
               System 2 Best          24125           100.00  
                        MACD          22812           100.00  
Zero Crossover System 1 Best          -5300             0.00  
                        MACD          -2650             0.00  
               System 2 Best            888           100.00  
                        MACD          -2650             0.00